# V18 BCR/TCR — Corrected Folder Scan + Full Analysis
**Fix:** Correct paths for gut_2021 supplementary and extracted files

In [ ]:
# Cell 1: Mount + Scan CORRECT folders
from google.colab import drive
drive.mount('/content/drive')
import os, glob
import pandas as pd
import numpy as np

PATHS = {
    'gut_2021 (docs)': '/content/drive/MyDrive/ITLAS/docs/gut_2021',
    'extracted (data)': '/content/drive/MyDrive/ITLAS/data/extracted',
}

all_found_files = []

for label, path in PATHS.items():
    print(f'\n{"="*70}')
    print(f'SCANNING: {label}')
    print(f'Path: {path}')
    print(f'{"="*70}')
    if not os.path.exists(path):
        print(f'  ⚠️ Path does not exist!')
        continue
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, '').count(os.sep)
        if depth > 3:
            continue
        indent = '  ' * (depth + 1)
        rel = os.path.relpath(root, path)
        if rel != '.':
            print(f'{indent}📁 {os.path.basename(root)}/')
        for f in sorted(files):
            fpath = os.path.join(root, f)
            fsize = os.path.getsize(fpath)
            if fsize > 1024*1024*1024:
                sz_str = f'{fsize/1024/1024/1024:.1f} GB'
            elif fsize > 1024*1024:
                sz_str = f'{fsize/1024/1024:.1f} MB'
            else:
                sz_str = f'{fsize/1024:.1f} KB'
            fl = f.lower()
            ext = os.path.splitext(f)[1].lower()
            flag = ''
            if any(kw in fl for kw in ['bcr','tcr','vdj','clone','contig','repertoire']):
                flag = ' ⭐ BCR/TCR'
            elif ext in ['.csv','.tsv','.txt','.xlsx']:
                flag = ' 📊'
            print(f'{indent}  {f} ({sz_str}){flag}')
            all_found_files.append({'path': fpath, 'name': f, 'size': fsize, 'ext': ext, 'source': label})

print(f'\n{"="*70}')
print(f'Total files found: {len(all_found_files)}')
bcr_tcr_files = [f for f in all_found_files if any(kw in f["name"].lower() for kw in ['bcr','tcr','vdj','clone','contig'])]
print(f'BCR/TCR related files: {len(bcr_tcr_files)}')
csv_files = [f for f in all_found_files if f['ext'] in ['.csv','.tsv','.txt']]
print(f'CSV/TSV/TXT files: {len(csv_files)}')

In [ ]:
# Cell 2: Preview ALL CSV/TSV files — identify BCR/TCR data
print(f'\n{"="*70}')
print('PREVIEWING ALL CSV/TSV FILES')
print(f'{"="*70}')

for finfo in csv_files:
    fpath = finfo['path']
    fname = finfo['name']
    sz = finfo['size']
    sz_str = f'{sz/1024:.1f} KB' if sz < 1024*1024 else f'{sz/1024/1024:.1f} MB'
    print(f'\n{"─"*60}')
    print(f'📄 {fname} ({sz_str}) — from {finfo["source"]}')
    try:
        # Auto-detect separator
        with open(fpath, 'r') as fh:
            first_line = fh.readline()
        sep = '\t' if '\t' in first_line else ','
        df = pd.read_csv(fpath, sep=sep, nrows=5, low_memory=False)
        print(f'   Columns ({len(df.columns)}): {list(df.columns[:15])}', end='')
        if len(df.columns) > 15:
            print(f'... +{len(df.columns)-15} more')
        else:
            print()
        # Full row count
        if sz < 100*1024*1024:  # only for files < 100MB
            n_rows = sum(1 for _ in open(fpath)) - 1
            print(f'   Rows: {n_rows:,}')
        # Check BCR/TCR relevance
        cols_str = ' '.join(c.lower() for c in df.columns)
        is_vdj = any(kw in cols_str for kw in 
                     ['bcr','tcr','vdj','clone','cdr3','v_gene','j_gene',
                      'chain','contig','barcode','igh','igk','igl','tra','trb',
                      'clonotype','productive'])
        if is_vdj:
            print(f'   ⭐⭐ BCR/TCR RELATED! ⭐⭐')
            print(f'   First 2 rows:')
            df2 = pd.read_csv(fpath, sep=sep, nrows=2, low_memory=False)
            print(df2.to_string())
    except Exception as e:
        print(f'   Error: {e}')

In [ ]:
# Cell 3: h5ad BCR deep analysis — Stage × Tissue × Donor
import scanpy as sc

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

print(f'{"="*70}')
print('BCR DEEP ANALYSIS: Stage × Tissue')
print(f'{"="*70}')

# BCR by Stage × Tissue
bcr_xt = obs.groupby(['Stage', 'tissue'], observed=True).agg(
    total=('BCR_clone.id', 'size'),
    bcr=('BCR_clone.id', lambda x: x.notna().sum()),
    unique_clones=('BCR_clone.id', lambda x: x.dropna().nunique()),
).reset_index()
bcr_xt['pct'] = (bcr_xt['bcr'] / bcr_xt['total'] * 100).round(1)
bcr_xt['pct_singleton'] = ((bcr_xt['unique_clones'] / bcr_xt['bcr']) * 100).round(1)
print('\n--- BCR cells by Stage × Tissue ---')
for stage in ['NL','IT','IA','AR','CR']:
    rows = bcr_xt[bcr_xt['Stage']==stage]
    for _, r in rows.iterrows():
        print(f'  {stage}/{r.tissue}: {int(r.bcr):,}/{int(r.total):,} ({r.pct}%), '
              f'unique={int(r.unique_clones):,}, singleton~{r.pct_singleton}%')

# Isotype by Stage × Tissue
print(f'\n--- Isotype % by Stage × Tissue ---')
for tissue_val in ['Liver', 'Blood']:
    print(f'\n  {tissue_val}:')
    for stage in ['NL','IT','IA','AR','CR']:
        sub = obs[(obs['Stage']==stage) & (obs['tissue']==tissue_val) & obs['BCR_CType'].notna()]
        if len(sub) == 0:
            print(f'    {stage}: no data')
            continue
        iso = sub['BCR_CType'].value_counts()
        total = iso.sum()
        parts = [f'{k}={v/total*100:.1f}%' for k, v in iso.items()]
        print(f'    {stage} (n={total}): {" | ".join(parts)}')

In [ ]:
# Cell 4: DONOR-LEVEL BCR metrics (Liver & Blood separately)
from scipy.stats import mannwhitneyu

print(f'{"="*70}')
print('DONOR-LEVEL BCR METRICS + Mann-Whitney NL→IT')
print(f'{"="*70}')

def donor_bcr(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'n_bcr':0,'pct_bcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,
                         'pct_switched':np.nan,'top_clone_size':0})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        if nu > 1:
            fr = cc.values / cc.sum()
            ent = -np.sum(fr * np.log2(fr))
            clon = 1 - ent/np.log2(nu)
        else:
            clon = 0
        iso = bcr['BCR_CType'].value_counts()
        it = iso.sum()
        igm = iso.get('IGHM',0)/it*100 if it>0 else np.nan
        igg = iso.get('IGHG',0)/it*100 if it>0 else np.nan
        iga = iso.get('IGHA',0)/it*100 if it>0 else np.nan
        switched = (igg if igg else 0) + (iga if iga else 0)
        rows.append({'Stage':stage,'donor':donor,'n_bcr':nb,
                     'pct_bcr':nb/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100 if nu>0 else np.nan,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,
                     'pct_switched':switched,'top_clone_size':cc.max()})
    return pd.DataFrame(rows)

for tissue_val in ['Liver','Blood']:
    df = donor_bcr(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — Donor-level BCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[df['Stage']==stage]
        if len(s)==0 or s['n_bcr'].sum()==0:
            print(f'  {stage}: no BCR data')
            continue
        valid = s[s['n_bcr']>0]
        print(f'  {stage} ({len(valid)} donors with BCR):')
        print(f'    BCR cells: {valid.n_bcr.mean():.0f}±{valid.n_bcr.std():.0f}')
        print(f'    Clonality: {valid.clonality.mean():.4f}±{valid.clonality.std():.4f}')
        print(f'    Singleton: {valid.pct_singleton.mean():.1f}%')
        print(f'    IgM: {valid.pct_IgM.mean():.1f}% | IgG: {valid.pct_IgG.mean():.1f}% | IgA: {valid.pct_IgA.mean():.1f}%')
        print(f'    Class-switched: {valid.pct_switched.mean():.1f}% | Top clone: {valid.top_clone_size.mean():.1f}')
    
    # Mann-Whitney: NL vs IT
    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  --- Mann-Whitney NL→IT ({tissue_val}) ---')
        for metric in ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_switched','pct_bcr']:
            nl_vals = nl[metric].dropna()
            it_vals = it[metric].dropna()
            if len(nl_vals)>=2 and len(it_vals)>=2:
                stat, p = mannwhitneyu(nl_vals, it_vals, alternative='two-sided')
                nl_m = nl_vals.mean()
                it_m = it_vals.mean()
                direction = '↑' if it_m > nl_m else '↓'
                pct_chg = ((it_m - nl_m) / nl_m * 100) if nl_m != 0 else float('inf')
                sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
                print(f'    {sig} {metric}: NL={nl_m:.2f} → IT={it_m:.2f} ({direction}{abs(pct_chg):.1f}%) p={p:.4f}')

In [ ]:
# Cell 5: DONOR-LEVEL TCR metrics + Mann-Whitney
print(f'{"="*70}')
print('DONOR-LEVEL TCR METRICS + Mann-Whitney NL→IT')
print(f'{"="*70}')

def donor_tcr(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'n_tcr':0,'pct_tcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,'top_clone':0})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        nu = len(cc)
        ns = (cc==1).sum()
        if nu > 1:
            fr = cc.values / cc.sum()
            ent = -np.sum(fr * np.log2(fr))
            clon = 1 - ent/np.log2(nu)
        else:
            clon = 0
        rows.append({'Stage':stage,'donor':donor,'n_tcr':nt,
                     'pct_tcr':nt/n*100,'clonality':clon,
                     'pct_singleton':ns/nu*100,'top_clone':cc.max()})
    return pd.DataFrame(rows)

for tissue_val in ['Liver','Blood']:
    df = donor_tcr(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — Donor-level TCR')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[df['Stage']==stage]
        if len(s)==0 or s['n_tcr'].sum()==0:
            print(f'  {stage}: no TCR data')
            continue
        valid = s[s['n_tcr']>0]
        print(f'  {stage} ({len(valid)} donors with TCR):')
        print(f'    TCR cells: {valid.n_tcr.mean():.0f}±{valid.n_tcr.std():.0f}')
        print(f'    Clonality: {valid.clonality.mean():.4f}±{valid.clonality.std():.4f}')
        print(f'    Singleton: {valid.pct_singleton.mean():.1f}%')
        print(f'    Top clone: {valid.top_clone.mean():.0f}')
    
    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  --- Mann-Whitney NL→IT ({tissue_val}) ---')
        for metric in ['clonality','pct_singleton','pct_tcr','top_clone']:
            nl_v = nl[metric].dropna()
            it_v = it[metric].dropna()
            if len(nl_v)>=2 and len(it_v)>=2:
                stat, p = mannwhitneyu(nl_v, it_v, alternative='two-sided')
                nl_m, it_m = nl_v.mean(), it_v.mean()
                d = '↑' if it_m > nl_m else '↓'
                pct = ((it_m-nl_m)/nl_m*100) if nl_m!=0 else float('inf')
                sig = '★' if p<0.05 else '†' if p<0.10 else ' '
                print(f'    {sig} {metric}: NL={nl_m:.3f} → IT={it_m:.3f} ({d}{abs(pct):.1f}%) p={p:.4f}')

In [ ]:
# Cell 6: BCR × Lineage + V-gene IT vs NL + Isotype shift
print(f'{"="*70}')
print('BCR × LINEAGE  |  V-GENE USAGE  |  ISOTYPE SHIFT')
print(f'{"="*70}')

# BCR by lineage
print('\n--- BCR+ cells by lineage ---')
bcr_lin = obs[obs['BCR_clone.id'].notna()].groupby('major_lineage', observed=True).size()
print(bcr_lin.sort_values(ascending=False).to_string())

# TCR by lineage
print('\n--- TCR+ cells by lineage ---')
tcr_lin = obs[obs['TCR_clone.id'].notna()].groupby('major_lineage', observed=True).size()
print(tcr_lin.sort_values(ascending=False).to_string())

# BCR V-gene top5 by Stage × Tissue
print(f'\n--- BCR V-gene Top5 by Stage × Tissue ---')
for tissue_val in ['Liver','Blood']:
    print(f'\n  {tissue_val}:')
    for stage in ['NL','IT','IA','AR']:
        sub = obs[(obs['Stage']==stage)&(obs['tissue']==tissue_val)&obs['BCR_v_gene'].notna()]
        if len(sub)==0:
            continue
        vg = sub['BCR_v_gene'].value_counts()
        tot = vg.sum()
        top5 = [(g, f'{c/tot*100:.1f}%') for g,c in vg.head(5).items()]
        print(f'    {stage} (n={tot}): {top5}')

# BCR subcluster detail (B and PlasmaB)
print(f'\n--- BCR+ by B/PlasmaB subcluster × Stage ---')
b_plasma = obs[(obs['BCR_clone.id'].notna()) & 
               (obs['major_lineage'].isin(['B','PlasmaB']))]
sc_stage = b_plasma.groupby(['gut2021_subcluster_v2','Stage'], observed=True).size().unstack(fill_value=0)
for col_order in [['NL','IT','IA','AR','CR']]:
    present = [c for c in col_order[0] if c in sc_stage.columns]
    sc_stage = sc_stage[present]
print(sc_stage.to_string())

In [ ]:
# Cell 7: Save all results
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)

# Donor-level BCR
bcr_all = pd.concat([donor_bcr(obs,'Liver'), donor_bcr(obs,'Blood')], ignore_index=True)
bcr_all.insert(2, 'tissue', bcr_all.index.map(lambda i: 'Liver' if i < len(donor_bcr(obs,'Liver')) else 'Blood'))
# Simpler approach
bcr_liver_df = donor_bcr(obs, 'Liver')
bcr_liver_df['tissue'] = 'Liver'
bcr_blood_df = donor_bcr(obs, 'Blood')
bcr_blood_df['tissue'] = 'Blood'
bcr_all = pd.concat([bcr_liver_df, bcr_blood_df], ignore_index=True)
bcr_all.to_csv(f'{SAVE_DIR}/donor_level_BCR_metrics.csv', index=False)
print(f'Saved: donor_level_BCR_metrics.csv ({len(bcr_all)} rows)')

# Donor-level TCR
tcr_liver_df = donor_tcr(obs, 'Liver')
tcr_liver_df['tissue'] = 'Liver'
tcr_blood_df = donor_tcr(obs, 'Blood')
tcr_blood_df['tissue'] = 'Blood'
tcr_all = pd.concat([tcr_liver_df, tcr_blood_df], ignore_index=True)
tcr_all.to_csv(f'{SAVE_DIR}/donor_level_TCR_metrics.csv', index=False)
print(f'Saved: donor_level_TCR_metrics.csv ({len(tcr_all)} rows)')

print(f'\n✅ All saved to: {SAVE_DIR}')
print(f'\n{"="*70}')
print('KEY FINDINGS SO FAR')
print(f'{"="*70}')
print('''
1. h5ad contains 10 BCR/TCR columns directly — no external files needed
2. BCR: 12,764 cells (5.3%), heavy chain only (IGH)
   - Isotypes: IGHM (61%), IGHG (21%), IGHA (13%), IGHD
   - Class switching HAS occurred (~33% switched)
   - But clonal expansion is near-zero (>99% singleton)
3. TCR: 64,284 cells (26.5%), alpha chain only (TRA)
4. CR has ZERO BCR/TCR data (library not constructed?)
5. BCR V-gene: IGHV3-23 most frequent (known anti-HBs V gene)

→ "Switched but Stuck" confirmed: class switching complete,
   clonal expansion blocked. This is the B cell equivalent of
   the six-layer effector suppression in T/myeloid cells.
''')